# Checking InterLabel Train, Val and Test sets

Download it from here https://seq2fun.dcmb.med.umich.edu/InterLabelGO/InterLabelGO_data.tar.xz

In [ ]:
wget https://seq2fun.dcmb.med.umich.edu/InterLabelGO/InterLabelGO_data.tar.xz
tar -xvf InterLabelGO_data.tar.xz

ls InterLabelGO_data

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
go_terms = pd.read_csv('InterLabelGO_data/train/train_terms.tsv', sep='\t')
print(len(go_terms))

5177680


In [ ]:
go_terms

,EntryID,aspect,term
0,A0A009IHW8,BPO,GO:0006796
1,A0A009IHW8,BPO,GO:0019364
2,A0A009IHW8,BPO,GO:0044237
3,A0A009IHW8,BPO,GO:0019362
4,A0A009IHW8,BPO,GO:0072521
...,...,...,...
5177675,X6RM59,CCO,GO:0043229
5177676,X6RM59,CCO,GO:0005575
5177677,X6RM59,CCO,GO:0031974
5177678,X6RM59,CCO,GO:0005829


In [ ]:
go_terms = go_terms.rename(columns={'EntryID':'protein_id', 'term':'go_id', 'aspect': 'go_aspect'})

In [ ]:
from tqdm import tqdm
import pandas as pd

def collapse_go_terms_per_protein(df):
    go_fields = ["go_id", "go_aspect"]

    def agg_go_group(group):
        first = group.iloc[0].to_dict()

        go_ids = group["go_id"].tolist()  # Keep original list
        go_bp = group.loc[group["go_aspect"] == "BPO", "go_id"].tolist()
        go_mf = group.loc[group["go_aspect"] == "MFO", "go_id"].tolist()
        go_cc = group.loc[group["go_aspect"] == "CCO", "go_id"].tolist()

        out = {
            "protein_id": first["protein_id"],
            "go_ids": go_ids,
            "go_bp": go_bp if go_bp else np.nan,
            "go_mf": go_mf if go_mf else np.nan,
            "go_cc": go_cc if go_cc else np.nan,
        }

        # Add other fields from first row
        for col in df.columns:
            if col not in go_fields + ["protein_id"]:
                out[col] = first[col]

        return pd.Series(out)

    tqdm.pandas(desc="Collapsing GO terms")
    collapsed_df = df.groupby("protein_id").progress_apply(agg_go_group).reset_index(drop=True)
    return collapsed_df

In [ ]:
go_terms = collapse_go_terms_per_protein(go_terms)

Collapsing GO terms: 100%|██████████████████████████████████████████████████████████████████▊| 142141/142478 [00:33<00:00, 4343.92it/s]tqdm/std.py:917: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return getattr(df, df_function)(wrapper, **kwargs)
Collapsing GO terms: 100%|███████████████████████████████████████████████████████████████████| 142478/142478 [00:34<00:00, 4072.57it/s]


In [ ]:
go_terms

,protein_id,go_ids,go_bp,go_mf,go_cc
0,A0A009IHW8,"[GO:0006796, GO:0019364, GO:0044237, GO:001936...","[GO:0006796, GO:0019364, GO:0044237, GO:001936...","[GO:0003824, GO:0016798, GO:0016799, GO:001678...",NaN
1,A0A021WW32,"[GO:0010468, GO:0000278, GO:0010556, GO:002200...","[GO:0010468, GO:0000278, GO:0010556, GO:002200...",NaN,"[GO:0005634, GO:0098687, GO:0000794, GO:003198..."
2,A0A021WZA4,"[GO:0071944, GO:0005575, GO:0110165, GO:000588...",NaN,NaN,"[GO:0071944, GO:0005575, GO:0110165, GO:000588..."
3,A0A023FBW4,"[GO:0019958, GO:0019956, GO:0005488, GO:000367...",NaN,"[GO:0019958, GO:0019956, GO:0005488, GO:000367...",NaN
4,A0A023FBW7,"[GO:0019956, GO:0005488, GO:0003674, GO:000551...",NaN,"[GO:0019956, GO:0005488, GO:0003674, GO:000551...",NaN
...,...,...,...,...,...
142473,X6RLN4,"[GO:0005737, GO:0005575, GO:0110165, GO:000562...",NaN,NaN,"[GO:0005737, GO:0005575, GO:0110165, GO:000562..."
142474,X6RLP6,"[GO:0005634, GO:0043233, GO:0110165, GO:003198...",NaN,NaN,"[GO:0005634, GO:0043233, GO:0110165, GO:003198..."
142475,X6RLR1,"[GO:0005634, GO:0110165, GO:0031981, GO:000562...",NaN,NaN,"[GO:0005634, GO:0110165, GO:0031981, GO:000562..."
142476,X6RM59,"[GO:0005634, GO:0110165, GO:0031981, GO:000562...",NaN,NaN,"[GO:0005634, GO:0110165, GO:0031981, GO:000562..."


This is the InterLabel+ Unique number of Train Proteins

## Checking BioReason Cafa Data

In [ ]:
import pandas as pd
from datasets import load_dataset

# Load CAFA5 dataset
dataset = load_dataset("wanglab/cafa5", "cafa5_reasoning")

train_df_proteins = dataset['train']['protein_id']

In [ ]:
# Assume train_df_proteins is a DataFrame with a column 'protein_id'
# and go_terms is the DataFrame with columns ['EntryID', 'aspect', 'term']

train_proteins_set = set(train_df_proteins)
go_proteins_set = set(go_terms['protein_id'])

# 1️⃣ Proteins in both train_df and go_terms
overlap = train_proteins_set & go_proteins_set

# 2️⃣ Proteins in train_df but missing GO terms
missing_go = train_proteins_set - go_proteins_set

# 3️⃣ Proteins in go_terms but not in train_df (should be rare)
extra_go = go_proteins_set - train_proteins_set

# Stats
print("Total proteins in train_df:", len(train_proteins_set))
print("Total proteins with GO terms:", len(go_proteins_set))
print("Overlap proteins:", len(overlap))
print("Proteins in train_df but missing GO terms:", len(missing_go))
print("Proteins in go_terms but not in train_df:", len(extra_go))

# Optionally, list them
print("\nExample missing proteins (first 10):", list(missing_go)[:10])
print("Example extra proteins (first 10):", list(extra_go)[:10])

Total proteins in train_df: 133492
Total proteins with GO terms: 142478
Overlap proteins: 132638
Proteins in train_df but missing GO terms: 854
Proteins in go_terms but not in train_df: 9840

Example missing proteins (first 10): ['A0A411HBC7', 'A0A0G2JX72', 'A0A0R4IC37', 'Q51486', 'D3YZQ8', 'A0A8I5ZVU8', 'Q2V2P4', 'H3BMQ8', 'Q4WEY5', 'H0Y4E8']
Example extra proteins (first 10): ['P19409', 'A0A8M6Z3L2', 'A0A8M9PIQ9', 'A0A8M3B692', 'Q7K0F0', 'A0A8M2BJY4', 'A0A8M9QBU1', 'A3KP58', 'A0A8M2BJ10', 'A0A8M2BBK8']


In [ ]:
dataset['train']

Dataset({
    features: ['protein_id', 'protein_names', 'protein_function', 'organism', 'length', 'subcellular_location', 'sequence', 'go_ids', 'go_bp', 'go_mf', 'go_cc', 'structure_path', 'string_id', 'interaction_partners', 'interpro_ids', 'interpro_location', 'date_added', 'last_modified'],
    num_rows: 133492
})

# Creating Test Data from Interlabel+

### Adding in sequence

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
test_go_terms = pd.read_csv('InterLabelGO_data/test/0323_0224.tsv', sep='\t')
test_fasta = 'InterLabelGO_data/test/0323_0224.fasta'

In [ ]:
test_go_terms

,EntryID,term,aspect
0,A0A068CNX1,GO:0006725,BPO
1,A0A068CNX1,GO:0008150,BPO
2,A0A068CNX1,GO:0008152,BPO
3,A0A068CNX1,GO:0009058,BPO
4,A0A068CNX1,GO:0009698,BPO
...,...,...,...
230819,X6RLK1,GO:0043229,CCO
230820,X6RLK1,GO:0043231,CCO
230821,X6RLK1,GO:0043233,CCO
230822,X6RLK1,GO:0070013,CCO


In [ ]:
test_go_terms = test_go_terms.rename(columns={'EntryID':'protein_id', 'term':'go_id', 'aspect': 'go_aspect'})

In [ ]:
from tqdm import tqdm
import pandas as pd

def collapse_go_terms_per_protein(df):
    go_fields = ["go_id", "go_aspect"]

    def agg_go_group(group):
        first = group.iloc[0].to_dict()

        go_ids = group["go_id"].tolist()  # Keep original list
        go_bp = group.loc[group["go_aspect"] == "BPO", "go_id"].tolist()
        go_mf = group.loc[group["go_aspect"] == "MFO", "go_id"].tolist()
        go_cc = group.loc[group["go_aspect"] == "CCO", "go_id"].tolist()

        out = {
            "protein_id": first["protein_id"],
            "go_ids": go_ids,
            "go_bp": go_bp if go_bp else np.nan,
            "go_mf": go_mf if go_mf else np.nan,
            "go_cc": go_cc if go_cc else np.nan,
        }

        # Add other fields from first row
        for col in df.columns:
            if col not in go_fields + ["protein_id"]:
                out[col] = first[col]

        return pd.Series(out)

    tqdm.pandas(desc="Collapsing GO terms")
    collapsed_df = df.groupby("protein_id").progress_apply(agg_go_group).reset_index(drop=True)
    return collapsed_df

In [ ]:
test_go_terms = collapse_go_terms_per_protein(test_go_terms)

Collapsing GO terms: 100%|██████████| 8630/8630 [00:06<00:00, 1332.83it/s]


In [ ]:
from Bio import SeqIO

def add_fasta_sequences(df, fasta_path, id_col="protein_id", seq_col="sequence"):
    """
    Adds sequences from a FASTA file to a dataframe.
    Assumes FASTA headers match df[id_col] entries.
    """
    print(f"Reading sequences from {fasta_path} ...")
    seq_dict = {}
    for record in SeqIO.parse(fasta_path, "fasta"):
        seq_dict[record.id] = str(record.seq)

    print(f"Found {len(seq_dict)} sequences in FASTA.")
    
    # Map sequences to dataframe
    df[seq_col] = df[id_col].map(seq_dict)

    # Warn if any sequences are missing
    missing = df[seq_col].isna().sum()
    if missing > 0:
        print(f"⚠️ Warning: {missing} protein_ids from the dataframe were not found in the FASTA file.")

    return df

In [ ]:
test_go_terms = add_fasta_sequences(test_go_terms, test_fasta)

Reading sequences from InterLabelGO_data/test/0323_0224.fasta ...
Found 8630 sequences in FASTA.


In [ ]:
test_go_terms

,protein_id,go_ids,go_bp,go_mf,go_cc,sequence
0,A0A024R531,"[GO:0005575, GO:0005622, GO:0005634, GO:000565...",NaN,NaN,"[GO:0005575, GO:0005622, GO:0005634, GO:000565...",MPHNSIRSGHGGLNQLGGAFVNGRPLPEVVRQRIVDLAHQGVRPCD...
1,A0A068CNX1,"[GO:0006725, GO:0008150, GO:0008152, GO:000905...","[GO:0006725, GO:0008150, GO:0008152, GO:000905...","[GO:0003674, GO:0003824, GO:0016829, GO:001683...",NaN,MARLLLLLVGVLIACAAGARAGSEFLAEDNPIRQVVDGMHELESSI...
2,A0A072TK64,"[GO:0008150, GO:0010035, GO:0042221, GO:005089...","[GO:0008150, GO:0010035, GO:0042221, GO:005089...",NaN,NaN,MEENKKTMEKSVGFTEEQDALVVKSWNAMKKNSGDLSLKFFKKILE...
3,A0A072ULZ1,"[GO:0001666, GO:0006950, GO:0008150, GO:000960...","[GO:0001666, GO:0006950, GO:0008150, GO:000960...",NaN,NaN,MEENKKTVDGSVDFTEEQEALVVKSWNAMKNNSCDLSLKFFTKILE...
4,A0A075D5I4,"[GO:0006066, GO:0006807, GO:0008150, GO:000815...","[GO:0006066, GO:0006807, GO:0008150, GO:000815...","[GO:0003674, GO:0003824, GO:0008168, GO:000817...","[GO:0000325, GO:0005575, GO:0005622, GO:000573...",MAEKQQAVAEFYDNSTGAWEVFFGDHLHDGFYDPGTTATIAGSRAA...
...,...,...,...,...,...,...
8625,X1WHD4,"[GO:0008150, GO:0009719, GO:0009725, GO:001003...","[GO:0008150, GO:0009719, GO:0009725, GO:001003...",NaN,NaN,MSRILHPQVLTIWITLISRATADLTCDALLLLSTNLTARTLILWNQ...
8626,X5D907,"[GO:0005575, GO:0005622, GO:0005737, GO:000582...",NaN,NaN,"[GO:0005575, GO:0005622, GO:0005737, GO:000582...",MEELVVEVRGSNGAFYKAFVKDVHEDSITVAFENNWQPDRQIPFHD...
8627,X6RHX1,"[GO:0005575, GO:0005622, GO:0005737, GO:000578...",NaN,NaN,"[GO:0005575, GO:0005622, GO:0005737, GO:000578...",MGSENSALKSYTLREPPFTLPSGLAVYPAVLQDGKFASVFVYKREN...
8628,X6RK18,"[GO:0005575, GO:0005622, GO:0005634, GO:000565...",NaN,NaN,"[GO:0005575, GO:0005622, GO:0005634, GO:000565...",MAETREEETVSAEASGFSDLSDSEFLEFLDLEDAQESKALVNMPGP...


In [ ]:
print(
    f"""
Missing values:
---------------
protein_id: {test_go_terms['protein_id'].isna().sum()}
go_ids:     {test_go_terms['go_ids'].isna().sum()}
go_bp:      {test_go_terms['go_bp'].isna().sum()}
go_mf:      {test_go_terms['go_mf'].isna().sum()}
go_cc:      {test_go_terms['go_cc'].isna().sum()}
sequence:   {test_go_terms['sequence'].isna().sum()}
"""
)


Missing values:
---------------
protein_id: 0
go_ids:     0
go_bp:      2811
go_mf:      6550
go_cc:      5190
sequence:   0



In [ ]:
print(
    f"""
Unique values:
---------------
protein_id: {test_go_terms['protein_id'].nunique()}
sequence:   {test_go_terms['sequence'].nunique()}
"""
)


Unique values:
---------------
protein_id: 8630
sequence:   8528



In [ ]:
with open('interlabel_test_accessions.txt', 'w') as f:
    for acc in test_go_terms['protein_id']:
        f.write(acc + '\n')

## Downloading 2024-02 interlabel Data

wget https://ftp.ebi.ac.uk/pub/databases/uniprot/previous_releases/release-2024_02/knowledgebase/knowledgebase2024_02.tar.gz
tar -xzf knowledgebase2024_02.tar.gz

In [ ]:
import pandas as pd
import re

def parse_interlabel(file_path, protein_accessions):
    """
    Parse interlabel .dat file and extract metadata for selected protein accessions (AC line).

    Args:
        file_path (str): Path to the .dat file
        protein_accessions (list): List of accession IDs to extract (e.g., ['Q6GZX4', 'P69905'])

    Returns:
        pandas.DataFrame: Metadata for matching proteins
    """
    protein_accessions = set(protein_accessions)  # for fast lookup

    with open(file_path, 'r') as f:
        entries = f.read().split('//\n')

    data = []
    for entry in entries:
        if not entry.strip():
            continue

        # Extract all AC lines
        ac_matches = re.findall(r'^AC\s+(.+)', entry, re.MULTILINE)
        if not ac_matches:
            continue

        # Combine all ACs from possibly multiple AC lines
        entry_accessions = []
        for ac_line in ac_matches:
            entry_accessions.extend([a.strip() for a in ac_line.split(';') if a.strip()])

        # Check if any of the accessions are in our query list
        matched_accessions = [acc for acc in entry_accessions if acc in protein_accessions]
        if not matched_accessions:
            continue

        # For each accession that matched, extract metadata
        for matched_acc in matched_accessions:
            protein_name = None
            function = None
            organism = None
            subcellular_location = None

            # Protein name
            name_match = re.search(r'DE\s+RecName:\s+Full=(.*?);', entry)
            if name_match:
                protein_name = name_match.group(1).strip()

            # Function
            func_match = re.search(
                r'CC\s+-!-\s+FUNCTION:\s*(.*?)(?=\nCC\s+-!-|\nCC\s+-{3,}|$)',
                entry,
                re.DOTALL
            )
            if func_match:
                function = func_match.group(1).strip().replace('\nCC', '').replace('\n', ' ')

            # Organism
            org_match = re.search(r'OS\s+(.+)', entry)
            if org_match:
                organism = org_match.group(1).strip()

            # Subcellular location
            sub_match = re.search(
                r'CC\s+-!-\s+SUBCELLULAR LOCATION:\s*(.*?)(?=\nCC\s+-!-|\nCC\s+-{3,}|$)',
                entry,
                re.DOTALL
            )
            if sub_match:
                subcellular_location = sub_match.group(1).strip().replace('\nCC', '').replace('\n', ' ')

            data.append({
                'Accession': matched_acc,
                'Protein Name': protein_name,
                'protein_function': function,
                'organism': organism,
                'subcellular_location': subcellular_location
            })

    return pd.DataFrame(data)


# Example usage
if __name__ == "__main__":
    file_path = "uniprot/uniprot_sprot.dat"
    protein_accessions = list(test_go_terms['protein_id'])  # your query list
    df = parse_interlabel(file_path, protein_accessions)
    print(f"Done. Found {len(df)} entries.")
    df.to_csv("parsed_interlabel_interlabel_test.csv", index=False)

Done. Found 1286 entries.


In [ ]:
interlabel_test = pd.read_csv('parsed_interlabel_interlabel_test.csv')

In [ ]:
interlabel_test

,Accession,Protein Name,protein_function,organism,subcellular_location
0,C0HLS4,Pi-stichotoxin-Hmg5a {ECO:0000305},Toxin that inhibits rat ASIC3 channels (IC(50)...,Heteractis magnifica (Magnificent sea anemone)...,Secreted {ECO:0000269|PubMed:36287966}. ...
1,C0HM68,Pi/alpha-stichotoxin-Hmg5b {ECO:0000305},The non-oxidized toxin potentiates ACh-elicite...,Heteractis magnifica (Magnificent sea anemone)...,Secreted {ECO:0000269|PubMed:36287966}. ...
2,P35363,5-hydroxytryptamine receptor 2A,G-protein coupled receptor for 5-hydroxytrypta...,Mus musculus (Mouse).,Cell membrane {ECO:0000269|PubMed:21645528}; ...
3,Q9JJ16,5-hydroxytryptamine receptor 3B {ECO:0000305|P...,Forms serotonin (5-hydroxytryptamine/5-HT3)-ac...,Rattus norvegicus (Rat).,Postsynaptic cell membrane {ECO:0000250|...
4,D0NPN8,RxLR effector protein Avr3a {ECO:0000303|PubMe...,Multifunctional effector that can suppress hos...,Phytophthora infestans (strain T30-4) (Potato ...,"Secreted {ECO:0000269|PubMed:15894622, E..."
...,...,...,...,...,...
1281,P0CE46,Proton-coupled zinc antiporter SLC30A8 {ECO:00...,Proton-coupled zinc ion antiporter mediating t...,Rattus norvegicus (Rat).,"Cytoplasmic vesicle, secretory vesicle membran..."
1282,P53303,Zinc finger chaperone ZPR1 {ECO:0000305},Acts as a protein folding chaperone for elonga...,Saccharomyces cerevisiae (strain ATCC 204508 /...,Cytoplasm {ECO:0000269|PubMed:9852145}. Nucleu...
1283,A7KBS4,Zinc finger and SCAN domain containing protein 4D,Transcription factor required to regulate earl...,Mus musculus (Mouse).,Nucleus {ECO:0000255|PROSITE-ProRule:PRU00187}...
1284,Q3URS2,Zinc finger and SCAN domain containing protein 4F,Transcription factor required to regulate earl...,Mus musculus (Mouse).,Nucleus {ECO:0000255|PROSITE-ProRule:PRU00187}...


## Parsing Trembl Data to get remaining proteins metadata 2024-02

In [ ]:
tar -xzf knowledgebase2024_02.tar.gz
pigz -d uniprot_sprot.dat.gz
pigz -d uniprot_trembl.dat.gz

trembl uncompressed is ~850 gb so chunk it to make parsing it in this next step more efficient

In [ ]:
import os

input_file = "uniprot_trembl.dat"
output_dir = os.path.join("data", "trembl_chunks")
os.makedirs(output_dir, exist_ok=True)

chunk_size = 10000  # entries per file
entry_count = 0
chunk_index = 0

outfile = open(f"{output_dir}/chunk_{chunk_index:05d}.dat", "w")

with open(input_file, "r") as f:
    buffer = []
    for line in f:
        buffer.append(line)
        if line.strip() == "//":
            entry_count += 1
            outfile.writelines(buffer)
            buffer = []

            if entry_count % chunk_size == 0:
                outfile.close()
                chunk_index += 1
                outfile = open(f"{output_dir}/chunk_{chunk_index:05d}.dat", "w")

# write any remaining entries
if buffer:
    outfile.writelines(buffer)
outfile.close()

Search those chunks with an sbatch array job

In [ ]:
import sys, os, re, pandas as pd

# Usage: python parse_trembl_chunk_slurm.py <chunks_dir> <output_dir> <accessions_file> <array_idx> <num_chunks>

if len(sys.argv) < 6:
    print("Usage: python parse_trembl_chunk_slurm.py <chunks_dir> <output_dir> <accessions_file> <array_idx> <num_chunks>")
    sys.exit(1)

chunks_dir, output_dir, accessions_file, array_idx, num_chunks = sys.argv[1:6]
os.makedirs(output_dir, exist_ok=True)

# --- Load the accessions list ---
try:
    with open(accessions_file) as f:
        protein_accessions = set(line.strip() for line in f if line.strip())
except FileNotFoundError:
    print(f"Error: Accessions file not found at {accessions_file}")
    sys.exit(1)

# --- Helper Function for cleaning whitespace ---
def clean_text(text):
    """Replaces sequences of two or more whitespace characters with a single space."""
    if text is None:
        return None
    # Use re.sub to replace any one or more whitespace chars (\s+) with a single space
    return re.sub(r'\s+', ' ', text).strip()

def parse_chunk(file_path, protein_accessions):
    results = []
    with open(file_path, "r", encoding="utf-8") as f:
        entry = []
        for line in f:
            entry.append(line)
            if line.strip() == "//":
                text = "".join(entry)
                entry = []

                # Extract all accessions from AC lines
                acs = re.findall(r"^AC\s+(.+)", text, re.MULTILINE)
                if not acs:
                    continue
                accs = [a.strip() for l in acs for a in l.split(';') if a.strip()]

                # Keep only if any match our target set
                matches = [a for a in accs if a in protein_accessions]
                if not matches:
                    continue

                # --- METADATA EXTRACTION ---
                # 1. Try to extract the canonical Recommended Name (RecName)
                name_match = re.search(r"DE\s+RecName:\s+Full=(.*?);", text)
                
                # 2. If RecName is not found, fallback to SubName
                if not name_match:
                    name_match = re.search(r"DE\s+SubName:\s+Full=(.*?);", text)

                func_match = re.search(r"CC\s+-!-\s+FUNCTION:\s*(.*?)(?=\nCC\s+-!-|\nCC\s+-{3,}|$)", text, re.DOTALL)
                org_match  = re.search(r"OS\s+(.+)", text)
                sub_match  = re.search(r"CC\s+-!-\s+SUBCELLULAR LOCATION:\s*(.*?)(?=\nCC\s+-!-|\nCC\s+-{3,}|$)", text, re.DOTALL)
                
                # --- APPLY CLEANING & FORMATTING ---
                
                # Extract and clean Protein Name
                protein_name = clean_text(name_match.group(1)) if name_match else None
                
                # Extract and clean Function (initial cleanup handles raw line breaks/CC tags)
                func_raw = func_match.group(1).strip().replace("\nCC", "").replace("\n", " ") if func_match else None
                protein_function = clean_text(func_raw)
                
                # Extract and clean Organism
                organism = clean_text(org_match.group(1)) if org_match else None
                
                # Extract and clean Subcellular Location (initial cleanup handles raw line breaks/CC tags)
                sub_raw = sub_match.group(1).strip().replace("\nCC", "").replace("\n", " ") if sub_match else None
                subcellular_location = clean_text(sub_raw)

                for acc in matches:
                    results.append({
                        "Accession": acc,
                        "Protein Name": protein_name,
                        "protein_function": protein_function,
                        "organism": organism,
                        "subcellular_location": subcellular_location
                    })
    return pd.DataFrame(results)

# --- SLURM ARRAY LOGIC ---

# Convert array indices and chunk counts to integers
try:
    start_idx = int(array_idx) * int(num_chunks)
    num_chunks_int = int(num_chunks)
except ValueError:
    print("Error: array_idx and num_chunks must be valid integers.")
    sys.exit(1)

# Assuming the maximum number of chunks is 24824
MAX_CHUNKS = 24824 
end_idx = start_idx + num_chunks_int
if end_idx > MAX_CHUNKS:
    end_idx = MAX_CHUNKS

print(f"Parsing chunks from {start_idx} to {end_idx-1} (inclusive)")

for i in range(start_idx, end_idx):

    # --- Construct the correct chunk filename using zero-padding ---
    # :05d ensures the number is padded with leading zeros to 5 digits (e.g., 00001, 24824)
    chunk_file = os.path.join(chunks_dir, f"chunk_{i:05d}.dat")

    if not os.path.exists(chunk_file):
        print(f"⚠️  Chunk file not found: {chunk_file}. Exiting array job for this index.")
        # We use sys.exit(0) here as this is often desired behavior in Slurm array jobs 
        # when a file doesn't exist (e.g., if array index goes beyond available files).
        sys.exit(0)

    # --- Run parsing ---
    df = parse_chunk(chunk_file, protein_accessions)

    # --- Save results ---
    if not df.empty:
        # Create output filename from chunk name
        out_path = os.path.join(output_dir, f"results_{os.path.basename(chunk_file).replace('.dat', '.csv')}")
        df.to_csv(out_path, index=False)
        print(f"✅ Wrote {len(df)} entries to {out_path}")
    else:
        print(f"⚪ No matches found in {chunk_file}")

In [ ]:
!cat trembl_logs/* | grep "No matches found in" | wc -l

24606


In [ ]:
ls trembl_results/ | wc -l

218


24824 - 24606 = 218

Verified, that I covered all the chunks and I got all the proteins that could be found with this search

In [ ]:
import glob

# Match all files that start with 'parse_1279275_' and end with '.out'
files = sorted(glob.glob("trembl_results/results_chunk_*.csv"))

# Read and concat all, skipping the header row in each file
df = pd.concat(
    [pd.read_csv(f, sep=",", header=None, skiprows=1) for i, f in enumerate(files)],
    ignore_index=True
)

# (optional) if the first row of later chunks got concatenated as headers, fix them:
df.columns = ['Accession','Protein Name','protein_function','organism','subcellular_location']

print(f"✅ Combined {len(files)} files into a dataframe with {len(df):,} rows.")

✅ Combined 218 files into a dataframe with 7,344 rows.


In [ ]:
trembl_test = df
trembl_test

,Accession,Protein Name,protein_function,organism,subcellular_location
0,Q59VN4,Histone H4 {ECO:0000256|RuleBase:RU000528},Core component of nucleosome. Nucleosomes wrap...,Candida albicans (strain SC5314 / ATCC MYA-287...,Chromosome {ECO:0000256|ARBA:ARBA00004286}. Nu...
1,A0A2H0ZSM8,Phosphodiesterase {ECO:0000256|RuleBase:RU363067},NaN,Candida auris (Yeast).,NaN
2,A0A2H0ZP18,Multidrug resistance protein {ECO:0008006|Goog...,NaN,Candida auris (Yeast).,Membrane {ECO:0000256|ARBA:ARBA00004141}; Mult...
3,A0A2H0ZEH6,BZIP domain-containing protein {ECO:0000259|PR...,NaN,Candida auris (Yeast).,Cytoplasm {ECO:0000256|ARBA:ARBA00004496}. Nuc...
4,A0A2H0ZMI1,Ras-like protein 1 {ECO:0000313|EMBL:PIS51845.1},NaN,Candida auris (Yeast).,NaN
...,...,...,...,...,...
7339,A0A8V1APS5,Otopetrin 3 {ECO:0000313|Ensembl:ENSGALP000100...,NaN,Gallus gallus (Chicken).,Cell membrane {ECO:0000256|ARBA:ARBA00004651};...
7340,A0A1L8GVF0,DNA repair protein RAD50 {ECO:0000313|RefSeq:X...,NaN,Xenopus laevis (African clawed frog).,"Chromosome, telomere {ECO:0000256|ARBA:ARBA000..."
7341,A0A1L8FS35,Transcription factor AP-2-alpha {ECO:0000256|A...,NaN,Xenopus laevis (African clawed frog).,Nucleus {ECO:0000256|ARBA:ARBA00004123}.
7342,Q3B8L1,MSX1 protein {ECO:0000313|EMBL:AAI06247.1},NaN,Xenopus laevis (African clawed frog).,"Nucleus {ECO:0000256|ARBA:ARBA00004123, ECO:00..."


In [ ]:
interlabel_test = pd.concat([interlabel_test,trembl_test])

In [ ]:
interlabel_test

,Accession,Protein Name,protein_function,organism,subcellular_location
0,C0HLS4,Pi-stichotoxin-Hmg5a {ECO:0000305},Toxin that inhibits rat ASIC3 channels (IC(50)...,Heteractis magnifica (Magnificent sea anemone)...,Secreted {ECO:0000269|PubMed:36287966}. ...
1,C0HM68,Pi/alpha-stichotoxin-Hmg5b {ECO:0000305},The non-oxidized toxin potentiates ACh-elicite...,Heteractis magnifica (Magnificent sea anemone)...,Secreted {ECO:0000269|PubMed:36287966}. ...
2,P35363,5-hydroxytryptamine receptor 2A,G-protein coupled receptor for 5-hydroxytrypta...,Mus musculus (Mouse).,Cell membrane {ECO:0000269|PubMed:21645528}; ...
3,Q9JJ16,5-hydroxytryptamine receptor 3B {ECO:0000305|P...,Forms serotonin (5-hydroxytryptamine/5-HT3)-ac...,Rattus norvegicus (Rat).,Postsynaptic cell membrane {ECO:0000250|...
4,D0NPN8,RxLR effector protein Avr3a {ECO:0000303|PubMe...,Multifunctional effector that can suppress hos...,Phytophthora infestans (strain T30-4) (Potato ...,"Secreted {ECO:0000269|PubMed:15894622, E..."
...,...,...,...,...,...
7339,A0A8V1APS5,Otopetrin 3 {ECO:0000313|Ensembl:ENSGALP000100...,NaN,Gallus gallus (Chicken).,Cell membrane {ECO:0000256|ARBA:ARBA00004651};...
7340,A0A1L8GVF0,DNA repair protein RAD50 {ECO:0000313|RefSeq:X...,NaN,Xenopus laevis (African clawed frog).,"Chromosome, telomere {ECO:0000256|ARBA:ARBA000..."
7341,A0A1L8FS35,Transcription factor AP-2-alpha {ECO:0000256|A...,NaN,Xenopus laevis (African clawed frog).,Nucleus {ECO:0000256|ARBA:ARBA00004123}.
7342,Q3B8L1,MSX1 protein {ECO:0000313|EMBL:AAI06247.1},NaN,Xenopus laevis (African clawed frog).,"Nucleus {ECO:0000256|ARBA:ARBA00004123, ECO:00..."


In [ ]:
interlabel_test.columns

Index(['Accession', 'Protein Name', 'protein_function', 'organism',
       'subcellular_location'],
      dtype='object')

In [ ]:
interlabel_test = interlabel_test.rename(columns={'Accession':'protein_id', 'Protein Name':'protein_names'})

In [ ]:
test_go_terms

,protein_id,go_ids,go_bp,go_mf,go_cc,sequence
0,A0A024R531,"[GO:0005575, GO:0005622, GO:0005634, GO:000565...",NaN,NaN,"[GO:0005575, GO:0005622, GO:0005634, GO:000565...",MPHNSIRSGHGGLNQLGGAFVNGRPLPEVVRQRIVDLAHQGVRPCD...
1,A0A068CNX1,"[GO:0006725, GO:0008150, GO:0008152, GO:000905...","[GO:0006725, GO:0008150, GO:0008152, GO:000905...","[GO:0003674, GO:0003824, GO:0016829, GO:001683...",NaN,MARLLLLLVGVLIACAAGARAGSEFLAEDNPIRQVVDGMHELESSI...
2,A0A072TK64,"[GO:0008150, GO:0010035, GO:0042221, GO:005089...","[GO:0008150, GO:0010035, GO:0042221, GO:005089...",NaN,NaN,MEENKKTMEKSVGFTEEQDALVVKSWNAMKKNSGDLSLKFFKKILE...
3,A0A072ULZ1,"[GO:0001666, GO:0006950, GO:0008150, GO:000960...","[GO:0001666, GO:0006950, GO:0008150, GO:000960...",NaN,NaN,MEENKKTVDGSVDFTEEQEALVVKSWNAMKNNSCDLSLKFFTKILE...
4,A0A075D5I4,"[GO:0006066, GO:0006807, GO:0008150, GO:000815...","[GO:0006066, GO:0006807, GO:0008150, GO:000815...","[GO:0003674, GO:0003824, GO:0008168, GO:000817...","[GO:0000325, GO:0005575, GO:0005622, GO:000573...",MAEKQQAVAEFYDNSTGAWEVFFGDHLHDGFYDPGTTATIAGSRAA...
...,...,...,...,...,...,...
8625,X1WHD4,"[GO:0008150, GO:0009719, GO:0009725, GO:001003...","[GO:0008150, GO:0009719, GO:0009725, GO:001003...",NaN,NaN,MSRILHPQVLTIWITLISRATADLTCDALLLLSTNLTARTLILWNQ...
8626,X5D907,"[GO:0005575, GO:0005622, GO:0005737, GO:000582...",NaN,NaN,"[GO:0005575, GO:0005622, GO:0005737, GO:000582...",MEELVVEVRGSNGAFYKAFVKDVHEDSITVAFENNWQPDRQIPFHD...
8627,X6RHX1,"[GO:0005575, GO:0005622, GO:0005737, GO:000578...",NaN,NaN,"[GO:0005575, GO:0005622, GO:0005737, GO:000578...",MGSENSALKSYTLREPPFTLPSGLAVYPAVLQDGKFASVFVYKREN...
8628,X6RK18,"[GO:0005575, GO:0005622, GO:0005634, GO:000565...",NaN,NaN,"[GO:0005575, GO:0005622, GO:0005634, GO:000565...",MAETREEETVSAEASGFSDLSDSEFLEFLDLEDAQESKALVNMPGP...


In [ ]:
final_interlabel_test = pd.merge(interlabel_test, test_go_terms, on='protein_id')

In [ ]:
print(
    f"""
Missing values:
---------------
protein_id:           {final_interlabel_test['protein_id'].isna().sum()}
protein_names:        {final_interlabel_test['protein_names'].isna().sum()}
protein_function:     {final_interlabel_test['protein_function'].isna().sum()}
organism:             {final_interlabel_test['organism'].isna().sum()}
subcellular_location: {final_interlabel_test['subcellular_location'].isna().sum()}
go_ids:               {final_interlabel_test['go_ids'].isna().sum()}
go_bp:                {final_interlabel_test['go_bp'].isna().sum()}
go_mf:                {final_interlabel_test['go_mf'].isna().sum()}
go_cc:                {final_interlabel_test['go_cc'].isna().sum()}
sequence:             {final_interlabel_test['sequence'].isna().sum()}
"""
)


Missing values:
---------------
protein_id:           0
protein_names:        0
protein_function:     6528
organism:             0
subcellular_location: 4268
go_ids:               0
go_bp:                2811
go_mf:                6550
go_cc:                5190
sequence:             0



In [ ]:
print(
    f"""
Unique values:
---------------
protein_id:      {final_interlabel_test['protein_id'].nunique()}
sequence:        {final_interlabel_test['sequence'].nunique()}
protein_names:   {final_interlabel_test['protein_names'].nunique()}
"""
)


Unique values:
---------------
protein_id:      8630
sequence:        8528
protein_names:   6851



In [ ]:
final_interlabel_test['protein_names'] = final_interlabel_test['protein_names'].str.replace(r'\s?\{.*?\}', '', regex=True).str.strip()
final_interlabel_test['protein_function'] = final_interlabel_test['protein_function'].str.replace(r'\s?\{.*?\}|\s?\((?:Pub).*?\)', '', regex=True).str.strip()
final_interlabel_test['organism'] = final_interlabel_test['organism'].str.replace(r'\.+$', '', regex=True).str.strip()
final_interlabel_test['subcellular_location'] = final_interlabel_test['subcellular_location'].str.replace(r'\s?\{.*?\}|\s?\((?:Pub).*?\)', '', regex=True).str.strip()

In [ ]:
final_interlabel_test['protein_names'] = final_interlabel_test['protein_names'].str.replace(r'\s+', ' ', regex=True).str.strip()
final_interlabel_test['protein_function'] = final_interlabel_test['protein_function'].str.replace(r'\s+', ' ', regex=True).str.strip()
final_interlabel_test['organism'] = final_interlabel_test['organism'].str.replace(r'\s+', ' ', regex=True).str.strip()
final_interlabel_test['subcellular_location'] = final_interlabel_test['subcellular_location'].str.replace(r'\s+', ' ', regex=True).str.strip()

In [ ]:
final_interlabel_test['protein_names'] = final_interlabel_test['protein_names'].str.replace(r'[.\s]+$', '', regex=True).str.strip()
final_interlabel_test['protein_function'] = final_interlabel_test['protein_function'].str.replace(r'[.\s]+$', '', regex=True).str.strip()
final_interlabel_test['organism'] = final_interlabel_test['organism'].str.replace(r'[.\s]+$', '', regex=True).str.strip()
final_interlabel_test['subcellular_location'] = final_interlabel_test['subcellular_location'].str.replace(r'[.\s]+$', '', regex=True).str.strip()

In [ ]:
final_interlabel_test.to_csv('temp_final_interlabel_test.csv', header=True, index=False)

Manually fixed 5 organism names

In [ ]:
final_interlabel_test = pd.read_csv('temp_final_interlabel_test.csv')

In [ ]:
from tqdm import tqdm
from goatools.obo_parser import GODag
import pandas as pd
import numpy as np

# Load ontology
print("Loading Gene Ontology structure...")
# Make sure the path is correct for your environment
obodag = GODag("../data/go-basic.obo") 

# Precompute ALL ancestor terms for all GO terms
print("Precomputing ALL ancestor terms...")
go_to_ancestors = {}
for go_id, term in tqdm(obodag.items(), desc="Building ancestor map"):
    
    # --- CORRECTED ANCESTOR LOOKUP for older goatools ---
    
    # 1. Get ALL ancestors (parents, grandparents, etc.) - this returns a SET of GO ID STRINGS,
    #    but does NOT include the term itself in older versions.
    all_ancestors_no_self = term.get_all_parents() 
    
    # 2. Manually create the set that includes the term itself.
    all_ancestors_with_self = set(all_ancestors_no_self)
    all_ancestors_with_self.add(go_id)
    
    # 3. Store the set of all ancestors EXCLUDING the term itself.
    #    This is the set of terms that MUST be present for the annotation to be complete.
    go_to_ancestors[go_id] = all_ancestors_with_self - {go_id}
    
# ----------------------------------------------------------------------
# MODIFIED CHECKING FUNCTION (NO CHANGES NEEDED HERE)
# ----------------------------------------------------------------------

def check_ancestor_completeness(go_terms):
    """
    Checks if a list of GO terms contains ALL of the ancestors for every term in the list.
    
    Returns: 
        True if all ancestors are present, False otherwise.
    """
    if not isinstance(go_terms, (list, set)) or not go_terms:
        return True # Treat empty/invalid input as complete (no missing ancestors)
    
    # 1. Get the set of all unique ancestors required by the original terms
    required_ancestors = set()
    for go_id in go_terms:
        # NOTE: If a GO_ID is invalid (not in obodag), it won't be in the map.
        if go_id in go_to_ancestors:
            required_ancestors.update(go_to_ancestors[go_id])
            
    # 2. The annotation is complete if the required ancestors are a subset of the terms provided.
    terms_set = set(go_terms)
    missing_ancestors = required_ancestors - terms_set
    
    # Return True if nothing is missing, False if ancestors are missing
    return not bool(missing_ancestors)


# ----------------------------------------------------------------------
# APPLY TO DATAFRAME AND COUNT (NO CHANGES NEEDED HERE)
# ----------------------------------------------------------------------

print("\nChecking GO Ancestor Completeness...")
tqdm.pandas(desc="Checking GO Ancestor Completeness")

# For 'go_ids' column
# This applies the check and returns True/False for each row.
completeness_check = final_interlabel_test['go_ids'].progress_apply(check_ancestor_completeness)

# Count the number of rows where the check returned False (i.e., ancestors are missing)
missing_ancestors_count = (~completeness_check).sum()

print(f"\nTotal rows checked: {len(final_interlabel_test)}")
print(f"Number of rows in 'go_ids' with MISSING ancestors: {missing_ancestors_count}")
print(f"Percentage of incomplete rows: {(missing_ancestors_count / len(final_interlabel_test)) * 100:.2f}%")

Loading Gene Ontology structure...
../data/go-basic.obo: fmt(1.2) rel(2023-01-01) 46,739 Terms
Precomputing ALL ancestor terms...


Building ancestor map: 100%|██████████| 46739/46739 [00:00<00:00, 51882.15it/s]



Checking GO Ancestor Completeness...


Checking GO Ancestor Completeness: 100%|██████████| 8630/8630 [00:00<00:00, 1166248.14it/s]


Total rows checked: 8630
Number of rows in 'go_ids' with MISSING ancestors: 0
Percentage of incomplete rows: 0.00%


In [ ]:
from goatools.obo_parser import GODag

# Load the ontology file (.obo). You can download it from http://purl.obolibrary.org/obo/go.obo
obo_file = "../data/go-basic.obo"
go_dag = GODag(obo_file)

# Build dictionary mapping GO term → depth
go_to_depth = {go_id: term.depth for go_id, term in go_dag.items()}

# List of GO columns to sort
go_columns = ["go_ids", "go_bp", "go_mf", "go_cc"]

# Function to sort a list of GO terms by depth
def sort_by_depth(go_list, go_to_depth):
    if isinstance(go_list, list) and go_list:
        # sort ascending by depth (shallowest first)
        return sorted(go_list, key=lambda go: go_to_depth[go])
    return go_list  # keep None or empty as-is

# Apply to all columns with a progress bar
from tqdm import tqdm
tqdm.pandas(desc="Sorting GO term columns")

for col in go_columns:
    final_interlabel_test[col] = final_interlabel_test[col].progress_apply(lambda x: sort_by_depth(x, go_to_depth))

../data/go-basic.obo: fmt(1.2) rel(2023-01-01) 46,739 Terms


Sorting GO term columns: 100%|██████████| 8630/8630 [00:00<00:00, 1238684.67it/s]


In [ ]:
import pandas as pd
import numpy as np

def validate_go_columns_fast(df, columns, go_to_depth):
    results = {}
    depth_lookup = go_to_depth  # local copy

    for col in columns:
        def check_sorted(go_list):
            # skip empty or invalid rows
            if not isinstance(go_list, (list, np.ndarray)) or not go_list:
                return True
            depths = [depth_lookup[go] for go in go_list]
            return all(earlier <= later for earlier, later in zip(depths, depths[1:]))

        results[col] = df[col].apply(check_sorted).all()
    return results

columns_to_check = ["go_ids", "go_bp", "go_mf", "go_cc"]
validation_results = validate_go_columns_fast(final_interlabel_test, columns_to_check, go_to_depth)

print("All GO term columns sorted by depth:")
for col, is_valid in validation_results.items():
    print(f"{col}: {is_valid}")

All GO term columns sorted by depth:
go_ids: True
go_bp: True
go_mf: True
go_cc: True


In [ ]:
from tqdm import tqdm
tqdm.pandas(desc="Sanity check: GO categories sum to go_ids")

def check_go_sum(row):
    go_ids = set(row["go_ids"]) if isinstance(row["go_ids"], list) else set()
    go_union = set()
    for col in ["go_bp", "go_mf", "go_cc"]:
        if isinstance(row[col], list):
            go_union.update(row[col])
    return go_ids == go_union

# Apply to all rows
sanity_check_passed = final_interlabel_test.progress_apply(check_go_sum, axis=1).all()

print(f"Sanity check: All rows have go_bp + go_mf + go_cc == go_ids? {sanity_check_passed}")

# Optional: if you want to see which rows fail
failed_rows = final_interlabel_test[~final_interlabel_test.progress_apply(check_go_sum, axis=1)]
print(f"Number of rows failing sanity check: {len(failed_rows)}")

Sanity check: GO categories sum to go_ids: 100%|██████████| 8630/8630 [00:00<00:00, 60045.26it/s]


Sanity check: All rows have go_bp + go_mf + go_cc == go_ids? True


Sanity check: GO categories sum to go_ids: 100%|██████████| 8630/8630 [00:00<00:00, 88648.88it/s]

Number of rows failing sanity check: 0


In [ ]:
final_interlabel_test['length'] = final_interlabel_test['sequence'].str.len()

In [ ]:
from datasets import Dataset

test_hf = Dataset.from_pandas(final_interlabel_test, preserve_index=False)

In [ ]:
from datasets import DatasetDict

interlabel_test_dataset = DatasetDict({
    "test": test_hf
})

In [ ]:
interlabel_test_dataset

DatasetDict({
    test: Dataset({
        features: ['protein_id', 'protein_names', 'protein_function', 'organism', 'subcellular_location', 'go_ids', 'go_bp', 'go_mf', 'go_cc', 'sequence', 'length'],
        num_rows: 8630
    })
})

In [ ]:
interlabel_test_dataset.push_to_hub(
    "wanglab/cafa5",
    config_name="interlabel_test_dataset",
    commit_message="New Interlabel Test Dataset"
)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

CommitInfo(commit_url='https://huggingface.co/datasets/wanglab/cafa5/commit/cede5502af45c84a9950e8f2a04a45e802b80a8c', commit_message='New Interlabel Test Dataset', commit_description='', oid='cede5502af45c84a9950e8f2a04a45e802b80a8c', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/wanglab/cafa5', endpoint='https://huggingface.co', repo_type='dataset', repo_id='wanglab/cafa5'), pr_revision=None, pr_num=None)

# Download AlphaFold Structures for all Proteins

Follow instructions on setting up Google Cloud account and downloading from google cloud https://github.com/google-deepmind/alphafold/blob/main/afdb/README.md

In [ ]:
with open("alphafold_interlabel_test_manifest.txt", "w") as f:
    for protein in (list(final_interlabel_test['protein_id'])):
        f.write(f"gs://public-datasets-deepmind-alphafold-v4/AF-{protein}-F1-model_v4.cif\n")

```cat alphafold_interlabel_test_manifest.txt | gsutil -m cp -I af_db_interlabel/```

**Trying it a second way as only a quarter had structures from the above command**

In [ ]:
import requests
import os
from tqdm import tqdm

# Define the base URL structure
AF_BASE_URL = "https://alphafold.ebi.ac.uk/files/AF-{}-F1-model_v6.pdb"
DOWNLOAD_FOLDER = "alphafold_structures"

def download_alphafold_model(uniprot_accession, local_folder):
    """Constructs the URL and downloads the corresponding AlphaFold PDB file."""
    
    # 1. Construct the full download URL
    url = AF_BASE_URL.format(uniprot_accession)
    
    # 2. Define the local path and filename
    filename = url.split('/')[-1]
    local_path = os.path.join(local_folder, filename)
    
    # Create the folder if it doesn't exist
    os.makedirs(local_folder, exist_ok=True)
    
    print(f"\nAttempting to download {filename}...")
    
    try:
        # Use streaming request to avoid loading the entire file into memory
        with requests.get(url, stream=True) as r:
            r.raise_for_status() # Check for errors (e.g., 404 Not Found)
            
            # Get the total file size for the progress bar
            total_size = int(r.headers.get('content-length', 0))
            block_size = 8192 # Chunk size for streaming
            
            with open(local_path, 'wb') as f:
                with tqdm(
                    total=total_size, 
                    unit='iB', 
                    unit_scale=True, 
                    desc=filename, 
                    ncols=80
                ) as pbar:
                    for chunk in r.iter_content(chunk_size=block_size):
                        f.write(chunk)
                        pbar.update(len(chunk))
        
        print(f"✅ Successfully downloaded to: {local_path}")
        return local_path
        
    except requests.exceptions.HTTPError as e:
        print(f"❌ Error for {uniprot_accession}: File not found or failed to download (HTTP {e.response.status_code}).")
        return None
    except requests.exceptions.RequestException as e:
        print(f"❌ Network error for {uniprot_accession}: {e}")
        return None

# ----------------------------------------------------------------------
# EXAMPLE USAGE
# ----------------------------------------------------------------------

# Your list of UniProt accessions

with open("interlabel_test_accessions.txt", "r") as f:
    accession_list = [line.strip() for line in f if line.strip()]

# Loop through your list and download
for acc in accession_list:
    download_alphafold_model(acc, DOWNLOAD_FOLDER)

print(f"\nAll downloads attempted. Check the '{DOWNLOAD_FOLDER}' folder.")

### Counting number of structures

In [ ]:
!pwd

benchmarks


In [ ]:
import os

# --- Configuration ---
ACCESSION_FILE = "interlabel_test_accessions.txt"
DIR_ALPHAFOLD = "alphafold_structures" 

# --- Main Logic ---
print("Starting structure check...")

try:
    # 1. Load in the accession file
    with open(ACCESSION_FILE, 'r') as f:
        # Assuming one accession per line, stripping whitespace
        all_accessions = {line.strip() for line in f if line.strip()}
except FileNotFoundError:
    print(f"Error: Accession file '{ACCESSION_FILE}' not found.")
    exit()
except Exception as e:
    print(f"An error occurred while reading the accession file: {e}")
    exit()

total_accessions = len(all_accessions)
accessions_with_structures = set()
structures_found_count = 0

# 2. Check each accession for structures in the target directory
for accession in all_accessions:

    pdb_path = os.path.join(DIR_ALPHAFOLD, f"AF-{accession}-F1-model_v6.pdb")
    if os.path.exists(pdb_path):
        structures_found_count += 1
        accessions_with_structures.add(accession)
        
# --- Output Results ---
print("\n--- Structure Existence Report ---")
print(f"Total accessions loaded from '{ACCESSION_FILE}': {total_accessions}")
print(f"Directory checked: '{DIR_ALPHAFOLD}'")
print("-" * 40)

# Calculate unique accessions with structures and those without
unique_accessions_with_structures = len(accessions_with_structures)
accessions_without_structures = total_accessions - unique_accessions_with_structures

print(f"Accessions with a structure found: {unique_accessions_with_structures}")
print(f"Accessions with NO structure found: {accessions_without_structures}")
print("-" * 40)


Starting structure check...

--- Structure Existence Report ---
Total accessions loaded from 'interlabel_test_accessions.txt': 8630
Directory checked: 'alphafold_structures'
----------------------------------------
Accessions with a structure found: 6295
Accessions with NO structure found: 2335
----------------------------------------


In [ ]:
[c for c in all_accessions if c not in accessions_with_structures][:5]

['A0A8M3AW24', 'A0A8M9PJX8', 'H0Y9Y2', 'A0A6Q8PGB1', 'A0A8M9PFP4']

Switch to Bash terminal

In [ ]:
mkdir -p af_interlabel_shards

cp -r alphafold_structures af_interlabel_shards/
mv af_interlabel_shards/alphafold_structures af_interlabel_shards/shard_0

In [ ]:
tar -czvf af_interlabel_shards/shard_0.tar.gz af_interlabel_shards/shard_0

In [ ]:

git lfs install
git clone https://huggingface.co/datasets/wanglab/cafa5

cp -r af_interlabel_shards/ ../../cafa5/structures_interlabel/


git add structures_interlabel/
git commit -m "Add AlphaFold structures for interlabel proteins"
git push

# Adding Structure Path column

In [1]:
import pandas as pd

In [3]:
#testing if it works
from datasets import load_dataset

ds = load_dataset("wanglab/cafa5", "interlabel_test_dataset")

README.md: 0.00B [00:00, ?B/s]

interlabel_test_dataset/test-00000-of-00(…):   0%|          | 0.00/9.71M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/8630 [00:00<?, ? examples/s]

In [4]:
ds

DatasetDict({
    test: Dataset({
        features: ['protein_id', 'protein_names', 'protein_function', 'organism', 'subcellular_location', 'go_ids', 'go_bp', 'go_mf', 'go_cc', 'sequence', 'length', 'interpro_ids', 'interpro_location', 'structure_path'],
        num_rows: 8630
    })
})

In [5]:
interlabel_test = ds['test'].to_pandas()

In [6]:
with open("alphafold_structures_acc.txt", "r") as f:
    structure_accessions = [line.strip() for line in f if line.strip()]

In [7]:
len(structure_accessions)

6295

In [8]:
interlabel_test['structure_path'] = interlabel_test['protein_id'].apply(
    lambda x: f"AF-{x}-F1-model_v6.pdb" if x in structure_accessions else None
)

In [9]:
interlabel_test['structure_path'].isna().sum()

np.int64(2335)

In [10]:
from datasets import Dataset

# Assuming you already split your DataFrame earlier into train_df and test_df:
test_hf = Dataset.from_pandas(interlabel_test, preserve_index=False)

In [11]:
from datasets import DatasetDict

interlabel_dataset = DatasetDict({
    "test": test_hf
})

In [12]:
interlabel_dataset

DatasetDict({
    test: Dataset({
        features: ['protein_id', 'protein_names', 'protein_function', 'organism', 'subcellular_location', 'go_ids', 'go_bp', 'go_mf', 'go_cc', 'sequence', 'length', 'interpro_ids', 'interpro_location', 'structure_path'],
        num_rows: 8630
    })
})

In [13]:
interlabel_dataset.push_to_hub(
    "wanglab/cafa5",
    config_name="interlabel_test_dataset",
    commit_message="Fixed structure paths"
)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/datasets/wanglab/cafa5/commit/c79e097671a86ac6623802751513744a6cb9b8d4', commit_message='Fixed structure paths', commit_description='', oid='c79e097671a86ac6623802751513744a6cb9b8d4', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/wanglab/cafa5', endpoint='https://huggingface.co', repo_type='dataset', repo_id='wanglab/cafa5'), pr_revision=None, pr_num=None)

# Getting InterPro Domains for the Proteins

In [ ]:
import pandas as pd

In [ ]:
#testing if it works
from datasets import load_dataset

ds = load_dataset("wanglab/cafa5", "interlabel_test_dataset")

interlabel_test_dataset/test-00000-of-00(…):   0%|          | 0.00/9.13M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/8630 [00:00<?, ? examples/s]

In [ ]:
ds

DatasetDict({
    test: Dataset({
        features: ['protein_id', 'protein_names', 'protein_function', 'organism', 'subcellular_location', 'go_ids', 'go_bp', 'go_mf', 'go_cc', 'sequence', 'length'],
        num_rows: 8630
    })
})

In [ ]:
interlabel_test = ds['test'].to_pandas()

In [ ]:
from datasets import load_dataset
import requests
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from time import sleep
from tqdm import tqdm

def fetch_one_protein(accession, max_retries=3):
    url = f"https://www.ebi.ac.uk/interpro/api/entry/InterPro/protein/UniProt/{accession}"
    headers = {"Accept": "application/json"}

    for attempt in range(max_retries):
        try:
            r = requests.get(url, headers=headers, timeout=30)
            if r.status_code == 204:
                return []
            r.raise_for_status()

            results = r.json().get("results", [])
            rows = []

            for item in results:
                ipr_id = item.get("metadata", {}).get("accession")
                entry_name = item.get("metadata", {}).get("name")
                ipr_type = item.get("metadata", {}).get("type")

                all_fragments = []
                for loc in item.get("proteins", []):  # fixed: must go through item['proteins']
                    for entry in loc.get("entry_protein_locations", []):
                        all_fragments.extend(entry.get("fragments", []))

                if all_fragments:
                    start = min(f["start"] for f in all_fragments if "start" in f)
                    end = max(f["end"] for f in all_fragments if "end" in f)

                    rows.append({
                        "accession": accession,
                        "interpro_id": ipr_id,
                        "entry_name": entry_name,
                        "type": ipr_type,
                        "start": start,
                        "end": end,
                        "n_fragments": len(all_fragments)
                    })
            return rows

        except Exception as e:
            if attempt == max_retries - 1:
                print(f"❌ Failed for {accession}: {e}")
            sleep(2 * (attempt + 1))
    return []

def fetch_all_interpro_parallel(
    all_accessions,
    output_file="interpro_domains.tsv",
    interpro_lookup_file="interpro_lookup.tsv",
    max_threads=32
):
    protein_rows = []
    interpro_lookup = {}

    with ThreadPoolExecutor(max_workers=max_threads) as executor:
        futures = {executor.submit(fetch_one_protein, acc): acc for acc in all_accessions}
        for future in tqdm(as_completed(futures), total=len(futures), desc="Fetching InterPro"):
            rows = future.result()
            protein_rows.extend(rows)

    # Extract lookup table
    for row in protein_rows:
        ipr = row["interpro_id"]
        if ipr not in interpro_lookup:
            interpro_lookup[ipr] = {
                "entry_name": row["entry_name"],
                "type": row["type"]
            }

    # Save data
    pd.DataFrame(protein_rows).to_csv(output_file, sep="\t", index=False)
    pd.DataFrame([
        {"interpro_id": ipr, "entry_name": val["entry_name"], "type": val["type"]}
        for ipr, val in interpro_lookup.items()
    ]).to_csv(interpro_lookup_file, sep="\t", index=False)

    print(f"✅ Done. Fetched {len(protein_rows)} entries across {len(all_accessions)} proteins")

# ---------- Main execution ----------
if __name__ == "__main__":
    ds = load_dataset("wanglab/cafa5", "interlabel_test_dataset")
    interlabel_test = ds['test'].to_pandas()

    proteins = set(list(interlabel_test['protein_id']))

    print(f"Fetched {len(proteins)} proteins")

    fetch_all_interpro_parallel(
        list(proteins),
        output_file="interlabel_interpro_domains.tsv",
        interpro_lookup_file="interlabel_interpro_lookup.tsv",
        max_threads=32
    )

In [ ]:
interpro = pd.read_csv('interlabel_interpro_domains.tsv', sep='\t')

In [ ]:
interpro

,accession,interpro_id,entry_name,type,start,end,n_fragments
0,A0A6Q8PHC7,IPR002013,SAC domain,domain,93,504,1
1,A0A6Q8PHC7,IPR043573,Polyphosphoinositide phosphatase Fig4-like,family,11,762,1
2,A0A804HL99,IPR000313,PWWP domain,domain,329,442,1
3,A0A804HL99,IPR049583,"Peregrin, PWWP domain",domain,329,448,1
4,A6HB00,IPR014505,"UMP-CMP kinase 2, mitochondrial",family,40,416,1
...,...,...,...,...,...,...,...
41310,A6JLB5,IPR035899,Dbl homology (DH) domain superfamily,homologous_superfamily,1029,1242,1
41311,A6JLB5,IPR036034,PDZ superfamily,homologous_superfamily,837,929,1
41312,A6JLB5,IPR040655,"TIAM1, CC-Ex domain",domain,571,668,1
41313,A6JLB5,IPR043537,Tiam1/Tiam2/Protein still life,family,3,1589,1


In [ ]:
set(interpro['type'])

{'active_site',
 'binding_site',
 'conserved_site',
 'domain',
 'family',
 'homologous_superfamily',
 'ptm',
 'repeat'}

In [ ]:
from collections import defaultdict

# Define your custom type priority
type_order = {
    'homologous_superfamily': 0,
    'family': 1,
    'domain': 2,
    'conserved_site': 3,
    'ptm': 4,
    'repeat': 5,
    'active_site': 6,
    'binding_site': 7
}

# Intermediate mapping
accession_to_entries = defaultdict(list)

# Store everything per accession
for acc, ipr, start, end, typ in zip(
    interpro["accession"], interpro["interpro_id"],
    interpro["start"], interpro["end"], interpro["type"]
):
    accession_to_entries[acc].append((ipr, start, end, typ))

# Final output structure
collapsed_data = []

for acc, entries in accession_to_entries.items():
    interpro_ids = set()
    interpro_location = {}
    
    # Sort by (type priority, start)
    entries_sorted = sorted(
        entries,
        key=lambda x: (type_order.get(x[3], 99), x[1])  # (type, start)
    )

    for ipr, start, end, typ in entries_sorted:
        if ipr in interpro_location:
            raise ValueError(f"InterPro ID {ipr} appears more than once for protein {acc}")
        interpro_ids.add(ipr)
        interpro_location[ipr] = (start, end)

    collapsed_data.append({
        "accession": acc,
        "interpro_ids": list(interpro_ids),
        "interpro_location": interpro_location
    })

# Convert to DataFrame
collapsed_df = pd.DataFrame(collapsed_data)

In [ ]:
collapsed_df

,accession,interpro_ids,interpro_location
0,A0A6Q8PHC7,"[IPR002013, IPR043573]","{'IPR043573': (11, 762), 'IPR002013': (93, 504)}"
1,A0A804HL99,"[IPR049583, IPR000313]","{'IPR000313': (329, 442), 'IPR049583': (329, 4..."
2,A6HB00,"[IPR014505, IPR039430, IPR027417]","{'IPR027417': (218, 416), 'IPR014505': (40, 41..."
3,A0A8M2BE60,"[IPR031867, IPR021802, IPR011598, IPR036638]","{'IPR036638': (217, 311), 'IPR031867': (4, 142..."
4,Q3TAD2,"[IPR052251, IPR013087, IPR057986]","{'IPR052251': (15, 282), 'IPR057986': (15, 142..."
...,...,...,...
7193,A0A8M9QHK6,"[IPR050599, IPR031688, IPR005821, IPR031649, I...","{'IPR027359': (70, 1252), 'IPR005446': (106, 6..."
7194,A0A7I2V2Y6,"[IPR011989, IPR041123, IPR013598, IPR014877, I...","{'IPR011989': (1, 1010), 'IPR016024': (28, 987..."
7195,A0A8M6YYV3,"[IPR035436, IPR043145, IPR002017, IPR018159, I...","{'IPR036872': (7, 263), 'IPR036020': (2956, 29..."
7196,A6K1N1,"[IPR047544, IPR051628, IPR002867, IPR047545, I...","{'IPR013083': (554, 642), 'IPR051628': (111, 8..."


In [ ]:
interlabel_protein_ids = set(interlabel_test['protein_id'])

accessions_not_in_test = [
    c for c in interlabel_protein_ids
    if c not in set(collapsed_df['accession'])
]

In [ ]:
len(accessions_not_in_test)

1432

# Update Train and Test with Interpro Data

In [ ]:
interpro_collapsed = collapsed_df.rename(columns={"accession": "protein_id"})

interlabel_test = interlabel_test.merge(interpro_collapsed, on="protein_id", how="left")

In [ ]:
print("Rows where test interpro is None:", interlabel_test["interpro_ids"].isnull().sum())

Rows where test interpro is None: 1432


## Interpro Metadata

In [ ]:
interpro_metadata = pd.read_csv('interlabel_interpro_lookup.tsv', sep='\t')

In [ ]:
interpro_metadata

,interpro_id,entry_name,type
0,IPR002013,SAC domain,domain
1,IPR043573,Polyphosphoinositide phosphatase Fig4-like,family
2,IPR000313,PWWP domain,domain
3,IPR049583,"Peregrin, PWWP domain",domain
4,IPR014505,"UMP-CMP kinase 2, mitochondrial",family
...,...,...,...
7445,IPR047546,"E3 ubiquitin-protein ligase RNF216, Rcat domain",domain
7446,IPR058758,"E3 ubiquitin-protein ligase RNF216, UBA domain",domain
7447,IPR040655,"TIAM1, CC-Ex domain",domain
7448,IPR043537,Tiam1/Tiam2/Protein still life,family


**QC to check if every interpro ID in interlabel reasoning is in this metadata df**

In [ ]:
# Step 1: create a set of valid InterPro IDs
valid_ids = set(interpro_metadata["interpro_id"].dropna())

# Step 2: explode interpro_ids into separate rows
exploded = interlabel_test[["interpro_ids"]].explode("interpro_ids")

# Step 3: filter out null values first
exploded = exploded[exploded["interpro_ids"].notna()]

# Step 4: mark invalid IDs
exploded["is_invalid"] = ~exploded["interpro_ids"].isin(valid_ids)

# Step 5: count invalids
invalid_count = exploded["is_invalid"].sum()

print(invalid_count if invalid_count > 0 else 0)

0


**All interpro ids in interlabel dataframes are in this metadata df**

# Upload Data

In [ ]:
import json

In [ ]:
#Making it into a json object so that huggingface can handle it
interlabel_test["interpro_location"] = interlabel_test["interpro_location"].apply(json.dumps)

In [ ]:
from datasets import Dataset

test_hf = Dataset.from_pandas(interlabel_test, preserve_index=False)

In [ ]:
from datasets import DatasetDict

interlabel_dataset = DatasetDict({
    "test": test_hf
})

In [ ]:
interlabel_dataset

DatasetDict({
    test: Dataset({
        features: ['protein_id', 'protein_names', 'protein_function', 'organism', 'subcellular_location', 'go_ids', 'go_bp', 'go_mf', 'go_cc', 'sequence', 'length', 'interpro_ids', 'interpro_location'],
        num_rows: 8630
    })
})

In [ ]:
interlabel_dataset.push_to_hub(
    "wanglab/cafa5",
    config_name="interlabel_test_dataset",
    commit_message="Uploaded InterPro domains with all metadata, including start and end and type"
)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/9 [00:00<?, ?ba/s]

Uploading...:   0%|          | 0.00/9.95M [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/datasets/wanglab/cafa5/commit/24930cdce8e9e0ef85ad2468a17cfbff705b754a', commit_message='Uploaded InterPro domains with all metadata, including start and end and type', commit_description='', oid='24930cdce8e9e0ef85ad2468a17cfbff705b754a', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/wanglab/cafa5', endpoint='https://huggingface.co', repo_type='dataset', repo_id='wanglab/cafa5'), pr_revision=None, pr_num=None)

In [ ]:
from datasets import Dataset

# Ensure the columns that contain dictionaries or lists are JSON-serializable
interpro_hf = Dataset.from_pandas(interpro_metadata, preserve_index=False)

In [ ]:
from datasets import DatasetDict

interlabel_dataset = DatasetDict({
    "metadata": interpro_hf,
})

In [ ]:
interlabel_dataset

DatasetDict({
    metadata: Dataset({
        features: ['interpro_id', 'entry_name', 'type'],
        num_rows: 7450
    })
})

In [ ]:
interlabel_dataset.push_to_hub(
    "wanglab/cafa5",
    config_name="interlabel_interpro_metadata",
    commit_message="Upload the InterPro interlabel Metadata, downloaded from the Uniprot Cross references"
)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/8 [00:00<?, ?ba/s]

Uploading...:   0%|          | 0.00/206k [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

CommitInfo(commit_url='https://huggingface.co/datasets/wanglab/cafa5/commit/e4819bbef690d5c42db6fc709db9b9b018de19d6', commit_message='Upload the InterPro interlabel Metadata, downloaded from the Uniprot Cross references', commit_description='', oid='e4819bbef690d5c42db6fc709db9b9b018de19d6', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/wanglab/cafa5', endpoint='https://huggingface.co', repo_type='dataset', repo_id='wanglab/cafa5'), pr_revision=None, pr_num=None)